# NAFP + NMFP Recipe Retrain — 10 Epochs (no Drive needed)

**Pre-registered protocol:** `data/results/nafp/recipe_v2_10ep/PROTOCOL.md` on user's Mac.

**Goal:** Train NAFP on FMA-medium with 2 NMFP recipe fixes (F_MIN=160, one-anchor-per-track sampler) for exactly 10 epochs.

**How to use:**
1. Runtime → Change runtime type → GPU (T4 or better)
2. Run cells in order (Shift+Enter for each, OR Runtime → Run all)
3. Cell 2 will prompt you to upload 2 files (kaggle.json + the patched code tarball)
4. Wait ~60-90 min total
5. At the end, download `/content/recipe_v2_output.tar.gz` via the file panel

**IMPORTANT:** Do NOT close the tab during training. Colab disconnects inactive sessions.

## 1. GPU sanity check

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert out.returncode == 0, 'No GPU. Runtime > Change runtime type > select T4 GPU'
print('GPU:', out.stdout.strip())

## 2. Locate the 2 files you uploaded via the sidebar

**Before running this cell**, upload these 2 files to `/content/` via the **left sidebar** (folder icon → upload button):

- `kaggle.json` (your Kaggle API token from kaggle.com → Settings → API → Create New Token)
- `nafp_patched_recipe_v2.tar.gz` (445 KB, from your Mac at `/Users/prita/Desktop/Audio Fingerprinting/afp_bench/`)

This cell finds them automatically (handles Colab's `(1)` suffix duplicates).

In [ ]:
import os, shutil, glob

kj = sorted(glob.glob('/content/kaggle*.json'))
tb = sorted(glob.glob('/content/*recipe_v2*.tar.gz'))
assert kj, 'No kaggle.json in /content/. Upload via left sidebar.'
assert tb, 'No tarball in /content/. Upload nafp_patched_recipe_v2.tar.gz via sidebar.'

print(f'kaggle creds:   {kj[0]}  ({os.path.getsize(kj[0])} bytes)')
print(f'patched repo:   {tb[0]}  ({os.path.getsize(tb[0])/1024:.1f} KB)')

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(kj[0], '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if tb[0] != '/content/nafp_patched_recipe_v2.tar.gz':
    shutil.copy(tb[0], '/content/nafp_patched_recipe_v2.tar.gz')

print('\nReady. Continue with next cell.')

## 3. Install Python dependencies (~2-3 min)

In [ ]:
%pip install -q 'tensorflow==2.19.0' 'tf-keras==2.19.0' 'kapre==0.3.7' 'numpy<2.2' kaggle pyyaml librosa
import tensorflow; print('tensorflow:', tensorflow.__version__)
import kapre; print('kapre:', kapre.__version__)
import tf_keras; print('tf_keras: OK')

## 4. Download NAFP training data from Kaggle (~5-10 min, ~30 GB)

In [ ]:
import subprocess, os, glob
os.makedirs('/content/data', exist_ok=True)
out = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'mimbres/neural-audio-fingerprint',
     '-p', '/content/data', '--unzip'],
    capture_output=True, text=True, timeout=1800
)
print('STDOUT tail:', out.stdout[-1500:])
print('STDERR tail:', out.stderr[-1500:])
assert out.returncode == 0, 'Kaggle download failed. Check kaggle.json + dataset acceptance'

candidates = glob.glob('/content/data/**/music', recursive=True)
assert candidates, 'music/ subdir not found after download'
DATA_ROOT = os.path.dirname(candidates[0])
print(f'\nDATA_ROOT: {DATA_ROOT}')
for sub in ['music/train-10k-30s', 'aug/bg', 'aug/ir']:
    p = os.path.join(DATA_ROOT, sub)
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'  {sub}: {n} entries')

## 5. Extract patched NAFP code + verify patches

In [ ]:
import tarfile, os, shutil
REPO_ROOT = '/content/nafp_upstream'
if os.path.exists(REPO_ROOT):
    shutil.rmtree(REPO_ROOT)
with tarfile.open('/content/nafp_patched_recipe_v2.tar.gz', 'r:gz') as tar:
    tar.extractall('/content/')
os.rename('/content/upstream', REPO_ROOT)
print(f'Extracted to {REPO_ROOT}')

with open(f'{REPO_ROOT}/config/recipe_v2.yaml') as f:
    cfg_txt = f.read()
for must_have in ['F_MIN : 160.', "TR_SEG_MODE : 'random_oneshot'", 'MAX_EPOCH : 10']:
    assert must_have in cfg_txt, f'PATCH MISSING: {must_have!r}'
print('All 3 patches verified in recipe_v2.yaml.')

## 6. Rewrite config DIR paths for Colab + tighten knobs

In [ ]:
import yaml
cfg_path = f'{REPO_ROOT}/config/recipe_v2.yaml'
with open(cfg_path) as f: cfg = yaml.safe_load(f)
cfg['DIR']['SOURCE_ROOT_DIR'] = f'{DATA_ROOT}/music/'
cfg['DIR']['BG_ROOT_DIR']     = f'{DATA_ROOT}/aug/bg/'
cfg['DIR']['IR_ROOT_DIR']     = f'{DATA_ROOT}/aug/ir/'
cfg['DIR']['SPEECH_ROOT_DIR'] = f'{DATA_ROOT}/aug/speech/common_voice_8k/en/'
cfg['DIR']['OUTPUT_ROOT_DIR'] = '/content/logs/emb/'
cfg['DIR']['LOG_ROOT_DIR']    = '/content/logs/'
cfg.setdefault('DATA_SEL', {})['REDUCE_ITEMS_P'] = 0
cfg.setdefault('TRAIN', {})['MINI_TEST_IN_TRAIN'] = False
with open(cfg_path, 'w') as f: yaml.dump(cfg, f, sort_keys=False)
print('Config patched. Key values:')
print(f"  MAX_EPOCH:    {cfg['TRAIN']['MAX_EPOCH']}")
print(f"  F_MIN:        {cfg['MODEL']['F_MIN']}")
print(f"  TR_SEG_MODE:  {cfg['DATA_SEL']['TR_SEG_MODE']}")
print(f"  TR_BATCH_SZ:  {cfg['BSZ']['TR_BATCH_SZ']}")
print(f"  TR_N_ANCHOR:  {cfg['BSZ']['TR_N_ANCHOR']}")

## 7. GPU compute sanity check

In [ ]:
import os, time
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
print('TF:', tf.__version__, 'Keras:', tf.keras.__name__)
gpus = tf.config.list_physical_devices('GPU')
assert gpus, 'NO GPU visible — aborting'
print(f'GPUs: {gpus}')
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print(f'  set_memory_growth failed: {e}')
with tf.device('/GPU:0'):
    a = tf.random.normal((2048, 2048)); b = tf.random.normal((2048, 2048))
    _ = tf.matmul(a, b).numpy()
    t0 = time.time(); _ = tf.matmul(a, b).numpy()
    ms = (time.time() - t0) * 1000
print(f'GPU matmul: {ms:.1f} ms (healthy: <30 ms, CPU fallback: >500 ms)')
assert ms < 200, f'Matmul too slow ({ms:.0f} ms) — TF likely on CPU. Aborting.'

## 8. Trainer compat patch (CosineDecay alias for TF 2.19)

In [ ]:
from pathlib import Path
trainer = Path(REPO_ROOT) / 'model' / 'trainer.py'
src = trainer.read_text()
if 'tf.keras.experimental.CosineDecay' in src:
    src = src.replace('tf.keras.experimental.CosineDecay',
                      'tf.keras.optimizers.schedules.CosineDecay')
    trainer.write_text(src)
    print('Patched trainer.py: CosineDecay alias')
else:
    print('CosineDecay alias already correct')

## 9. Train (10 epochs, ~50-80 min on T4)

Output streams below. Watch the per-step loss decrease.

In [ ]:
import subprocess, time, os
global TRAIN_START
TRAIN_START = time.time()
proc = subprocess.Popen(
    ['python', '-u', 'run.py', 'train', 'recipe_v2', '-c', 'recipe_v2',
     '--max_epoch=10'],
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
    env={**os.environ, 'TF_USE_LEGACY_KERAS': '1', 'PYTHONUNBUFFERED': '1'}
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - TRAIN_START
print(f'\nTraining finished in {elapsed/60:.1f} min, exit code {proc.returncode}')
assert proc.returncode == 0, 'Training failed — see output above'

## 10. Bundle output for download

After this cell, **download the file** via the Files panel on the left:
1. Click the folder icon on the left sidebar to open the Files panel
2. Find `recipe_v2_output.tar.gz` at the top level
3. Right-click → **Download**
4. Save to your Mac

In [ ]:
import tarfile, json, glob, os, shutil, time
ckpt_src = '/content/logs/checkpoint/recipe_v2'
assert os.path.exists(ckpt_src), f'Checkpoint dir missing: {ckpt_src}'
ckpt_files = glob.glob(f'{ckpt_src}/ckpt-*.index')
ckpt_indices = sorted([int(p.split('ckpt-')[1].split('.')[0]) for p in ckpt_files])
latest = ckpt_indices[-1] if ckpt_indices else None
assert latest is not None, 'No checkpoint files saved — training may have failed silently'

BUNDLE = '/content/recipe_v2_output'
if os.path.exists(BUNDLE): shutil.rmtree(BUNDLE)
os.makedirs(BUNDLE)
shutil.copytree(ckpt_src, f'{BUNDLE}/checkpoint')
shutil.copy(f'{REPO_ROOT}/config/recipe_v2.yaml', f'{BUNDLE}/recipe_v2.yaml')
manifest = {
    'ckpt_index': latest,
    'training_minutes': round((time.time() - TRAIN_START) / 60, 1) if 'TRAIN_START' in dir() else None,
    'config_F_MIN': 160.0,
    'config_TR_SEG_MODE': 'random_oneshot',
    'config_MAX_EPOCH': 10,
}
with open(f'{BUNDLE}/manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
with tarfile.open('/content/recipe_v2_output.tar.gz', 'w:gz') as tar:
    tar.add(BUNDLE, arcname='recipe_v2_output')
sz = os.path.getsize('/content/recipe_v2_output.tar.gz') / 1e6
print(f'\nBundled to /content/recipe_v2_output.tar.gz ({sz:.1f} MB)')
print(json.dumps(manifest, indent=2))
print('\n=== DONE ===')
print('NOW: open Files panel (folder icon on left sidebar) → right-click recipe_v2_output.tar.gz → Download')

## 11. (Optional) Auto-trigger download

Some Colab setups support `files.download` — try this if the file panel approach doesn't work.

In [ ]:
from google.colab import files
files.download('/content/recipe_v2_output.tar.gz')